# Arabic Sign Language Translator — Model Training

This notebook trains a CNN to classify 32 Arabic Sign Language gestures.

**Dataset:** ArASL_Database_54K_Final (~54K images, 32 classes)  
**Input shape:** 64×64×3  
**Output:** Saved model at `model/SL.h5`

> ⚠️ The dataset is not included in this repository. See README for download instructions.

## 1. Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

## 2. Configuration

In [ ]:
# Update this path to point to your local dataset
DATASET_ROOT = "path/to/ArASL_Database_54K_Final"
IMG_SIZE = 64
NUM_CLASSES = 32

# Map folder names to class indices
CLASS_FOLDERS = [
    'aleff', 'bb', 'taa', 'thaa', 'jeem', 'haa', 'khaa',
    'dal', 'thal', 'ra', 'zay', 'seen', 'sheen', 'saad',
    'dhad', 'ta', 'dha', 'ain', 'ghain', 'fa', 'gaaf',
    'kaaf', 'laam', 'meem', 'nun', 'ha', 'waw', 'ya',
    'toot', 'al', 'la', 'yaa'
]

LABELS = {
    0: 'أ',  1: 'ب',  2: 'ت',  3: 'ث',  4: 'ج',
    5: 'ح',  6: 'خ',  7: 'د',  8: 'ذ',  9: 'ر',
    10: 'ز', 11: 'س', 12: 'ش', 13: 'ص', 14: 'ض',
    15: 'ط', 16: 'ظ', 17: 'ع', 18: 'غ', 19: 'ف',
    20: 'ق', 21: 'ك', 22: 'ل', 23: 'م', 24: 'ن',
    25: 'ه', 26: 'و', 27: 'ي', 28: 'ة', 29: 'ال',
    30: 'لا', 31: 'ئ'
}

## 3. Load Dataset

In [ ]:
def load_images_from_folder(folder_path):
    """Load and resize all images from a folder."""
    images = []
    for filename in os.listdir(folder_path):
        img = cv2.imread(os.path.join(folder_path, filename))
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
    return images


def load_all_classes(dataset_root, class_folders):
    """Load all classes and return X, y arrays."""
    X, y = [], []
    for label_idx, folder in enumerate(class_folders):
        folder_path = os.path.join(dataset_root, folder)
        images = load_images_from_folder(folder_path)
        print(f"Class {label_idx:2d} ({folder:>6s}): {len(images)} images")
        X.extend(images)
        y.extend([label_idx] * len(images))
    return np.array(X), np.array(y)


X, y = load_all_classes(DATASET_ROOT, CLASS_FOLDERS)
print(f"\nTotal samples: {len(X)}, Shape: {X.shape}")

## 4. Preprocess & Split

In [ ]:
# Normalize pixel values to [0, 1]
X = X.astype('float32') / 255.0

# One-hot encode labels
y_encoded = to_categorical(y, num_classes=NUM_CLASSES)

# Train / validation split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Model Architecture

In [ ]:
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 4
    layers.Conv2D(64, (3, 3), activation='relu'),

    # Classifier
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.summary()

## 6. Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test)
)

## 7. Evaluate

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()

## 9. Save Model

In [ ]:
os.makedirs('model', exist_ok=True)
model.save('model/SL.h5')
print("Model saved to model/SL.h5")